In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Norm"
feature_cols = ["x1", "x2", "x3"]
df = pd.read_excel(file_path, sheet_name=sheet_name)
X = df[feature_cols].to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {
    "x_min": np.array([0.0, 0.0, 0.0]),   # 功效系数法每列下界
    "x_max": np.array([100.0, 100.0, 100.0])  # 每列上界
}

X_z = StandardScaler().fit_transform(X)     # 标准差法
X_mm = MinMaxScaler().fit_transform(X)       # 极差法
X_eff = 60 + 40*(X - params["x_min"]) / np.maximum(params["x_max"] - params["x_min"], 1e-12)

print(X_z[:3], X_mm[:3], X_eff[:3])


In [ ]:
"""
标准差法、极值差法、功效系数法

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "标准差法、极值差法、功效系数法.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
INDICATOR_COLUMNS = ["指标1", "指标2"]  # TODO: 请填写[指标列名列表]，说明：仅包含数值型指标。
LOWER_SCORE = 60  # TODO: 请填写[功效系数下限]，说明：常用 60。
UPPER_SCORE = 100  # TODO: 请填写[功效系数上限]，说明：常用 100。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    numeric = data[INDICATOR_COLUMNS].astype(float)
    z_score = (numeric - numeric.mean()) / numeric.std(ddof=1)
    min_max = (numeric - numeric.min()) / (numeric.max() - numeric.min())
    efficacy = LOWER_SCORE + (UPPER_SCORE - LOWER_SCORE) * min_max
    result = pd.concat(
        [z_score.add_suffix("_标准差法"), min_max.add_suffix("_极值差法"), efficacy.add_suffix("_功效系数法")],
        axis=1,
    )
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result.head())


if __name__ == "__main__":
    df = load_data()
    run_model(df)
